# SelfConditionedDenoisingAtoms — Examples

Three minimal examples:
1. **Load a pretrained model from HuggingFace**
2. **Instantiate a new untrained model**
3. **Forward pass with a QM9 sample**

## Example 1: Load a Pretrained Model from HuggingFace

Available pretrained models: `"ct-scd-pcq"`, `"ct-scd-amp"`

The checkpoint is downloaded from `Ty-Perez/<model_name>` on HuggingFace Hub and the bare `model` (`CET` or similar) is returned — ready for inference or fine-tuning.

In [9]:
from huggingface_hub import hf_hub_download
from models.model_helper import load_model
import os

# ── choose a pretrained model ──────────────────────────────────────────────────
REPO_PREFIX = "Ty-Perez/"
MODEL_NAME  = "ct-scd-pcq"          # or "ct-scd-amp" for materials
SAVE_DIR    = "./experiments"       

# create the save directory if it doesn't exist
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# 1. Download the checkpoint from HuggingFace (cached after first download)
ckpt_path = hf_hub_download(
    repo_id=f"{REPO_PREFIX}{MODEL_NAME}",
    filename="last.ckpt",
    cache_dir=SAVE_DIR,
)
print(f"Checkpoint at: {ckpt_path}")

# 2. Load the model weights
model, ema_model, ckpt = load_model(ckpt_path)
model.eval()

#print number of parameters in the model (in the millions)
num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Number of parameters: {num_params:.2f} M")
print(model)

'''
NOTE: for the pretrained models on huggingface the ema_model is NOT trained. 
Do not use the ema_model. The training script provided also allows for training
with an ema model if needed, however we did not this to be necessary for good performance.
'''


Checkpoint at: ./experiments/models--Ty-Perez--ct-scd-pcq/snapshots/fcd4353d3b9f90eb38e08ccc0050cea02623d85f/last.ckpt
Loading weights from restart checkpoint
--- MODEL SUCESSFULLY LOADED ---
Number of parameters: 10.06 M
CET(
  (prior_model): ModuleList()
  (noise_head): EquivariantVector(
    (output_network): ModuleList(
      (0): GatedEquivariantBlock(
        (vec1_proj): Linear(in_features=256, out_features=256, bias=False)
        (vec2_proj): Linear(in_features=256, out_features=128, bias=False)
        (update_net): MLP(
          (act): SiLU()
          (layers): Sequential(
            (0): Linear(in_features=512, out_features=256, bias=True)
            (1): SiLU()
            (2): Linear(in_features=256, out_features=256, bias=True)
          )
        )
        (act): SiLU()
      )
      (1): GatedEquivariantBlock(
        (vec1_proj): Linear(in_features=128, out_features=128, bias=False)
        (vec2_proj): Linear(in_features=128, out_features=1, bias=False)
        (

'\nNOTE: for the pretrained models on huggingface the ema_model is NOT trained. \nDo not use the ema_model. The training script provided also allows for training\nwith an ema model if needed, however we did not this to be necessary for good performance.\n'

## Example 2: Instantiate a New (Untrained) Model

`create_model` reads a model-config YAML and constructs the network from scratch with random weights. The YAML specifies the architecture (e.g. `CET`) and all hyper-parameters.

In [10]:
from models.model_helper import create_model

# Minimal args dict — only model_config is required.
# Any key present in the YAML can be overridden here.
args = {
    "model_config": "configs/model_configs/scd.yaml",  # architecture + hyper-params
    "prior_model": None,   # no prior (e.g. Atomref) by default
}

model = create_model(args)
model.eval()
print(model)

CET(
  (noise_head): EquivariantVector(
    (output_network): ModuleList(
      (0): GatedEquivariantBlock(
        (vec1_proj): Linear(in_features=256, out_features=256, bias=False)
        (vec2_proj): Linear(in_features=256, out_features=128, bias=False)
        (update_net): MLP(
          (act): SiLU()
          (layers): Sequential(
            (0): Linear(in_features=512, out_features=256, bias=True)
            (1): SiLU()
            (2): Linear(in_features=256, out_features=256, bias=True)
          )
        )
        (act): SiLU()
      )
      (1): GatedEquivariantBlock(
        (vec1_proj): Linear(in_features=128, out_features=128, bias=False)
        (vec2_proj): Linear(in_features=128, out_features=1, bias=False)
        (update_net): MLP(
          (act): SiLU()
          (layers): Sequential(
            (0): Linear(in_features=256, out_features=128, bias=True)
            (1): SiLU()
            (2): Linear(in_features=128, out_features=2, bias=True)
          )
    

## Example 3a: Forward Pass without torchnet graph kernel

In [11]:
import torch
from data.datasets import QM9
from torch_geometric.data import Batch
from data.datasets.transforms import AddStandardKeys
from torch_geometric.loader import DataLoader

#the AddStandardKeys Transform create the necessary keys for graph creation and standardization across different kinds of samples
dataset = QM9(root="tmp/qm9", dataset_arg="homo", transform=AddStandardKeys())
sample = dataset[0]
loader = DataLoader(dataset, batch_size=16, shuffle=True)
data_batch = next(iter(loader))

#forward pass on a batch
model.eval()
with torch.no_grad():
    out = model(
        z=data_batch.z,
        pos=data_batch.pos,
        batch=data_batch.batch,
        graph_batch=data_batch,
        return_atom_embs=True # set true to return atom level embeddings in the output dictionary
    )

print("Forward output keys:", sorted(out.keys()))
print("y shape:", tuple(out["y"].shape))
print('mol_emb shape:', tuple(out["mol_emb"].shape))
print('noise_pred shape:', tuple(out["noise_pred"].shape))
print("atom_embs shape:", tuple(out["atom_embs"].shape))


Forward output keys: ['atom_embs', 'mol_emb', 'noise_pred', 'y']
y shape: (16, 1)
mol_emb shape: (16, 256)
noise_pred shape: (293, 3)
atom_embs shape: (293, 256)


## Example 3: Forward Pass on a QM9 Sample with TorchMD-Net graph kernel
NOTE: This requires the TorchMD-Net graph kernel to be build. See instructions in the README.md

The model's `forward(z, pos, batch)` expects:
- `z` — atomic numbers `(N,)`
- `pos` — Cartesian coordinates `(N, 3)`
- `batch` — molecule index per atom `(N,)` (optional for a single molecule)

It returns a dict with keys `"y"` (scalar prediction), `"noise_pred"`, `"mol_emb"`, etc.

In [ ]:
import torch
from data.datasets import QM9
from torch_geometric.data import Batch

# ── 1. Load QM9 and grab one molecule ─────────────────────────────────────────
# dataset_arg selects the target property (see QM9.target_dict_custom for all options)
dataset = QM9(root="tmp/qm9", dataset_arg="homo")
sample = dataset[0]   # a single torch_geometric Data object

# Forward pass on a single sample
model.eval()
with torch.no_grad():
    out = model(z=sample.z, pos=sample.pos)

print("\nOutput keys:", list(out.keys()))

#forward pass on a batch 
batch = Batch.from_data_list([dataset[i] for i in range(4)])
with torch.no_grad():
    out_batch = model(z=batch.z, pos=batch.pos, batch=batch.batch)

#print keys in out_batch
print("Batched output keys:", list(out_batch.keys()))